In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone',num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("kdave/Indian_Financial_News")

In [ ]:
import pandas

df = ds['train'].to_pandas()
df

In [ ]:
from transformers import Trainer, TrainingArguments
df2 = df.copy()

In [ ]:
from sklearn.model_selection import train_test_split
df.columns

y = df2['Sentiment']
X = df2.drop(['URL','Summary','Sentiment'], axis=1)

In [ ]:
X_train_temp, X_temp, y_train_temp, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Split temporary set into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

#Compute metrics for evaluation during training. pred is a object containing predictions and labels, o/p is a dictionary of computed metrics  
def compute_metrics(pred):

    # Extract predictions and labels
    predictions = pred.predictions.argmax(-1)  # Take the index of the highest logit
    labels = pred.label_ids

    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:
import re
from transformers import pipeline, AutoTokenizer
from collections import Counter

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()

model_id = "/content/drive/MyDrive/RL_Project/finbert_kdave_trained/finbert_kdave_trained"

finbert = pipeline("sentiment-analysis", model=model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import re
from transformers import pipeline, AutoTokenizer
from collections import Counter

# saved model
model_id = "/content/drive/MyDrive/finbert_kdave_trained"

# Loading the sentiment analysis pipeline and the tokenizer.
model = AutoModelForSequenceClassification.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)


from datasets import load_dataset, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split


ds = load_dataset("kdave/Indian_Financial_News")
df = ds["train"].to_pandas()
X = df[["Content"]]
y = df["Sentiment"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# label mapping that we use
label_mapping = {"Positive": 2, "Neutral": 1, "Negative": 0}
y_val_mapped = y_val.map(label_mapping)

# Tokenizing and format val dataset
def preprocess_function(examples):
    return tokenizer(examples["Content"], padding="max_length", truncation=True, max_length=512)

val_dataset = Dataset.from_pandas(X_val.assign(labels=y_val))
val_dataset = val_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.remove_columns(["Content"])
val_dataset = val_dataset.map(lambda x: {"labels": label_mapping[x["labels"]]})

from datasets import Value
val_dataset = val_dataset.cast_column("labels", Value("int64"))

# use the trainer for prediction
trainer = Trainer(model = model)
predictions = trainer.predict(val_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

# to print confusion matrix
label_names = ["Negative", "Neutral", "Positive"]

print(classification_report(y_true, y_pred, target_names=label_names))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - FinBERT on Indian Financial News")
plt.show()
